# 20. Thiol 규칙 확장 (thiol_1, thiol_2)

## 이번 노트북에서 할 것
- thiol_1과 thiol_2가 화학적으로 어떻게 다른지 구조 확인
  (지방족 vs 방향족, 또는 다른 문맥 조건)
- 문헌 근거: 티올기의 금속결합/산화반응성 메커니즘 확인
- SMARTS 설계, 편집 방식(결합절단형 vs 원자편집형) 결정
- replacement_library.py 반영, 회귀 테스트

## 간략한 정리 (19까지)
- 라이브러리 12개 규칙 확정: nitro_group, aldehyde, Michael_acceptor_1,
  acid_halide, alkyl_halide, aniline, Sulfonic_acid_2, imine_1_oxime,
  imine_1_general, catechol, Thiocarbonyl_group, aniline_ring_bcp
- 편집 방식 5종류: 결합절단(fragment-cut), replace_element, add_substituent,
  reduce_bond, replace_ring(신규, BCP 고리치환)
- aniline_ring_bcp: 문헌 근거(방향족성 제거->RM/CYP-inhibition 감소->IADR 예방)
  기반, para-치환 아닐린의 벤젠고리를 BCP로 치환. BCO/NB/CUB는 근거 부족으로
  확장하지 않기로 결정(BCP만 검증된 확실한 전략)
- Git author 이메일 오류 3건 rebase로 소급 수정 완료 (force push 완료)
- test set은 여전히 미사용, valid set(seed=7 분할)으로만 개발/검증 중

## 다음에 해야 할 것 (오늘 끝나면)
- thiol 완료 후, 라이브러리 확장 마무리
- 최종 valid set 재검증(12개+신규 규칙 전부 포함): 커버리지, 성공률,
  3-endpoint 비교
- 이후 학생 승인 시 test set으로 단 1회 최종 검증

In [1]:
# 셀 1
!pip install rdkit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 31.1 MB/s eta 0:00:00


In [2]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 232, done.
remote: Counting objects: 100% (232/232), done.
remote: Compressing objects: 100% (162/162), done.
remote: Total 232 (delta 120), reused 163 (delta 64), pack-reused 0 (from 0)
Receiving objects: 100% (232/232), 618.21 KiB | 2.18 MiB/s, done.
Resolving deltas: 100% (120/120), done.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀 3
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀 4
import importlib
from rdkit import Chem
from rdkit.Chem import rdMMPA

import src.tools.replacement_library
import src.tools.molecule_editor
import src.tools.atom_editor
import src.tools.toxicophore_detector
import src.tools.agent

from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import find_core_and_target, reassemble_molecule, propose_fix, canonicalize, iterative_fix_loop
from src.tools.atom_editor import apply_atom_edit_from_rule

data = load_tox21_clean(random_state=7)
print("도구 로드 확인 완료 (valid set 사용, test set 보류)")

[06:25:12] WARNING: not removing hydrogen atom without neighbors
[06:25:13] Explicit valence for atom # 8 Al, 6, is greater than permitted
[06:25:13] Explicit valence for atom # 3 Al, 6, is greater than permitted
[06:25:13] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:25:14] Explicit valence for atom # 4 Al, 6, is greater than permitted
[06:25:15] Explicit valence for atom # 9 Al, 6, is greater than permitted
[06:25:15] Explicit valence for atom # 5 Al, 6, is greater than permitted
[06:25:15] Explicit valence for atom # 16 Al, 6, is greater than permitted
[06:25:16] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[06:25:17] WARNING: not removing hydrogen atom without neighbors


도구 로드 확인 완료 (valid set 사용, test set 보류)


In [5]:
examples_thiol = {}
for s in data['smiles_valid']:
    problems = detect_toxicophores(s)
    for p in problems:
        if p['rule_name'] in ["thiol_1", "thiol_2"] and p['rule_name'] not in examples_thiol:
            examples_thiol[p['rule_name']] = (s, p['atom_indices'])
    if len(examples_thiol) == 2:
        break

for name, (smi, indices) in examples_thiol.items():
    print(f"\n{name}: {smi}")
    mol = Chem.MolFromSmiles(smi)
    for idx in indices:
        atom = mol.GetAtomWithIdx(idx)
        print(f"  인덱스 {idx}: {atom.GetSymbol()} (방향족: {atom.GetIsAromatic()}, 이웃: {[n.GetSymbol() for n in atom.GetNeighbors()]})")


thiol_2: CCCCCCC(C)(C)S
  인덱스 9: S (방향족: False, 이웃: ['C'])

thiol_1: CC(C)OC(=S)[S-]
  인덱스 6: S (방향족: False, 이웃: ['C'])


In [6]:
pattern_thiol2 = Chem.MolFromSmarts("[SX2H1]")  # 단순 -SH
mol_t2 = Chem.MolFromSmiles("CCCCCCC(C)(C)S")
print("thiol_2 매치:", mol_t2.HasSubstructMatch(pattern_thiol2))
print("thiol_2 패턴 크기:", pattern_thiol2.GetNumAtoms())

pattern_thiol1 = Chem.MolFromSmarts("C(=S)[SX1-]")  # 디티오카바메이트 핵심부
mol_t1 = Chem.MolFromSmiles("CC(C)OC(=S)[S-]")
print("thiol_1 매치:", mol_t1.HasSubstructMatch(pattern_thiol1))
print("thiol_1 패턴 크기:", pattern_thiol1.GetNumAtoms())

thiol_2 매치: True
thiol_2 패턴 크기: 1
thiol_1 매치: True
thiol_1 패턴 크기: 3


In [12]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "Michael_acceptor_1": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=CC(=O)",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "saturated (C-C single bond)",
             "rationale": "알파,베타-불포화 카르보닐의 C=C 이중결합을 환원하여 "
                          "단백질 친전자성 부가반응(Michael addition, covalent "
                          "binding) 위험을 제거함"},
        ],
    },
    "acid_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
            {"smiles": "C(=O)O", "name": "ester",
             "rationale": "아마이드보다 극성이 낮고 유연한 대체 옵션, 가수분해 속도 조절 가능 (검증 필요)"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "problem_smarts": "[NH2]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "acetamide (acylated amine)",
             "rationale": "1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "반응성 아민을 제거하면서 전자끄는기로 고리 전자밀도 보정"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "S(=O)(=O)[OX2H1,OX1-]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "생리적 pH에서 이온화 정도(전하)를 크게 낮춰 세포막 투과성을 "
                          "개선함. 설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 "
                          "저해되는 경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
            {"smiles": "C(=O)O", "name": "carboxylic acid",
             "rationale": "설폰산보다 산성도가 약하고 부피가 작은 산성 bioisostere "
                          "(검증 필요)"},
        ],
    },
    "imine_1_oxime": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=N[OX2H1]",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
        ],
    },
    "imine_1_general": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=N",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "일반 이민(C=N-R)을 환원하여 가수분해 시 반응성 카르보닐로 "
                          "되돌아갈 수 있는 대사 불안정 경로를 제거함. 옥심 특유의 "
                          "메커니즘보다는 근거가 다소 약하며, 하위 구조별 개별 검증 필요"},
        ],
    },
    "catechol": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2H;$(Oc1ccccc1O)]",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "add_substituent", "param": "C", "name": "methoxy",
             "rationale": "인체의 COMT(catechol-O-methyltransferase) 효소가 카테콜을 "
                          "메톡시페놀로 메틸화하여 해독하는 생리적 경로와 동일한 원리. "
                          "오르토-퀴논으로의 산화 경로를 차단하여 세포독성/유전독성 우려를 "
                          "낮춤 (학생 확인 예정: ScienceDirect catechol overview, "
                          "PMC6643002 등 참고)"},
        ],
    },
    "Thiocarbonyl_group": {
        "edit_method": "atom_edit",
        "problem_smarts": "[#6]=[#16]",
        "target_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "carbonyl (O replacing S)",
             "rationale": "황을 산소로 대체(티오카르보닐->카르보닐)하는 것은 흔한 "
                          "bioisostere 전략으로, 갑상선 기능 저해 등 황 함유 작용기 "
                          "특유의 대사/독성 우려를 낮춤 (검증 필요, thiourea->urea "
                          "치환 논리와 동일 계열)"},
        ],
    },
    "aniline_ring_bcp": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NH2]c1ccc([#6,#7,#8,#16])cc1",
        "ring_atom_indices_in_pattern": [1, 2, 3, 4, 6, 7],
        "anchor_indices_in_pattern": (0, 5),
        "candidates": [
            {"edit_type": "replace_ring", "param": "[*:1]C12CC(C1)(C2)[*:2]",
             "name": "BCP (bicyclo[1.1.1]pentane)",
             "rationale": "para-이치환 아닐린의 방향족 벤젠 고리를 포화 bicyclic "
                          "탄소골격(BCP)으로 교체함. 방향족성 제거로 aniline reactive "
                          "metabolite(RM) 형성 및 CYP-inhibition을 감소시켜, 퀴논이민 "
                          "생성 경로를 차단하고 특이체질 약물 부작용(IADR) 위험을 낮춤 "
                          "(문헌 근거, 학생 제공). 벤젠과의 공간적 유사성, Fsp3 증가, "
                          "실제 성공 사례가 많아 우선 채택함(BCO/NB/CUB는 근거 부족으로 보류)"},
        ],
    },
    "thiol_2": {
        "problem_smarts": "[SX2H1]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "티올의 금속 킬레이팅 및 산화(이황화물/술펜산 형성) 반응성을 "
                          "제거하면서, 극성·수소결합 특성을 유사하게 유지함"},
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "티올을 아마이드로 대체하여 반응성을 낮추면서 약물유사 골격에서 "
                          "흔히 쓰이는 안정적 작용기로 전환 (검증 필요)"},
        ],
    },
    "thiol_1": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=S)[SX1-]",
        "candidates": [
            {"edit_type": "replace_multi",
             "param": [
                 {"idx_in_pattern": 1, "new_element": 8, "new_charge": 0},
                 {"idx_in_pattern": 2, "new_element": 7, "new_charge": 0},
             ],
             "name": "carbamate (O,N replacing S,S)",
             "rationale": "디티오카바메이트(R-O-C(=S)-S-)를 카바메이트(R-O-C(=O)-N)로 "
                          "전환. 두 황 원자를 각각 산소·질소로 교체하여 금속 킬레이팅 "
                          "능력과 효소 억제 활성(디티오카바메이트류 특유의 살충제성 "
                          "독성 기전)을 제거함 (검증 필요)"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [8]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix

test_thiol = "CCCCCCC(C)(C)S"
print("thiol_2 core_test:", find_core_and_target(test_thiol, "thiol_2"))
print("thiol_2 fix_test:", propose_fix(test_thiol, "thiol_2", candidate_idx=0))

# 회귀 테스트
print("\n=== 회귀 테스트 ===")
print(propose_fix("O=C(O)CCl", "alkyl_halide", candidate_idx=0))
print(propose_fix(example_para if 'example_para' in dir() else "Cc1cc(NS(=O)(=O)c2ccc(N)cc2)nc(C)n1", "aniline_ring_bcp", candidate_idx=0))

thiol_2 core_test: {'core': 'CCCCCCC(C)(C)[*:1]', 'target_removed': 'S[*:1]'}
thiol_2 fix_test: {'new_smiles': 'CCCCCCC(C)(C)O', 'candidate_used': 'hydroxyl (alcohol)', 'rationale': '티올의 금속 킬레이팅 및 산화(이황화물/술펜산 형성) 반응성을 제거하면서, 극성·수소결합 특성을 유사하게 유지함', 'is_valid': True}

=== 회귀 테스트 ===
{'new_smiles': 'O=C(O)CO', 'candidate_used': 'hydroxyl (alcohol)', 'rationale': '이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지', 'is_valid': True}
{'new_smiles': 'Cc1cc(NS(=O)(=O)C23CC(N)(C2)C3)nc(C)n1', 'candidate_used': 'BCP (bicyclo[1.1.1]pentane)', 'rationale': 'para-이치환 아닐린의 방향족 벤젠 고리를 포화 bicyclic 탄소골격(BCP)으로 교체함. 방향족성 제거로 aniline reactive metabolite(RM) 형성 및 CYP-inhibition을 감소시켜, 퀴논이민 생성 경로를 차단하고 특이체질 약물 부작용(IADR) 위험을 낮춤 (문헌 근거, 학생 제공). 벤젠과의 공간적 유사성, Fsp3 증가, 실제 성공 사례가 많아 우선 채택함(BCO/NB/CUB는 근거 부족으로 보류)', 'is_valid': True}


In [9]:
!git add src/tools/replacement_library.py
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   src/tools/replacement_library.py



In [10]:
!git commit -m "Add thiol_2 rule (simple aliphatic thiol -> alcohol/amide, metal chelation and oxidation reactivity reduction); thiol_1 (dithiocarbamate) deferred as separate future expansion given different chemistry"
!git push origin main

[main 7e8b9b5] Add thiol_2 rule (simple aliphatic thiol -> alcohol/amide, metal chelation and oxidation reactivity reduction); thiol_1 (dithiocarbamate) deferred as separate future expansion given different chemistry
 1 file changed, 13 insertions(+), 1 deletion(-)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 925 bytes | 925.00 KiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   c5ac4d0..7e8b9b5  main -> main


In [11]:
%%writefile src/tools/atom_editor.py
from rdkit import Chem


def apply_atom_edit_from_rule(smiles: str, rule_name: str, candidate_idx: int = 0):
    """replacement_library의 atom_edit 규칙을 이용해 원자/결합/고리 직접 편집을 수행."""
    from src.tools.replacement_library import get_replacement_candidates
    info = get_replacement_candidates(rule_name)
    if info is None or info.get("edit_method") != "atom_edit":
        return None
    if candidate_idx >= len(info["candidates"]):
        return None

    candidate = info["candidates"][candidate_idx]
    smarts = info["problem_smarts"]

    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    match = matches[0]

    rwmol = Chem.RWMol(mol)
    edit_type = candidate["edit_type"]

    if edit_type == "replace_element":
        target_idx = match[info["target_idx_in_pattern"]]
        atom = rwmol.GetAtomWithIdx(target_idx)
        atom.SetAtomicNum(candidate["param"])

    elif edit_type == "add_substituent":
        target_idx = match[info["target_idx_in_pattern"]]
        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(target_idx, offset, Chem.BondType.SINGLE)
        atom = rwmol.GetAtomWithIdx(target_idx)
        if atom.GetNumExplicitHs() > 0:
            atom.SetNumExplicitHs(atom.GetNumExplicitHs() - 1)
        else:
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_bond":
        idx1 = match[info["target_idx_pair_in_pattern"][0]]
        idx2 = match[info["target_idx_pair_in_pattern"][1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.SINGLE)
        for idx in (idx1, idx2):
            atom = rwmol.GetAtomWithIdx(idx)
            atom.SetNoImplicit(False)

    elif edit_type == "replace_multi":
        # param: [{"idx_in_pattern": int, "new_element": int, "new_charge": int}, ...]
        # 여러 원자를 한 번에, 각각 다른 원소/전하로 교체
        for sub in candidate["param"]:
            target_idx = match[sub["idx_in_pattern"]]
            atom = rwmol.GetAtomWithIdx(target_idx)
            atom.SetAtomicNum(sub["new_element"])
            atom.SetFormalCharge(sub.get("new_charge", 0))
            atom.SetNoImplicit(False)
            atom.SetNumExplicitHs(0)  # 새 원소 기준으로 암묵적 H를 다시 계산하도록 초기화

    elif edit_type == "replace_ring":
        ring_indices = [match[i] for i in info["ring_atom_indices_in_pattern"]]
        anchor_idx1 = match[info["anchor_indices_in_pattern"][0]]
        anchor_idx2 = match[info["anchor_indices_in_pattern"][1]]

        anchor1_ring_neighbor = None
        anchor2_ring_neighbor = None
        for ridx in ring_indices:
            ratom = mol.GetAtomWithIdx(ridx)
            neighbor_idxs = [n.GetIdx() for n in ratom.GetNeighbors()]
            if anchor_idx1 in neighbor_idxs:
                anchor1_ring_neighbor = ridx
            if anchor_idx2 in neighbor_idxs:
                anchor2_ring_neighbor = ridx

        if anchor1_ring_neighbor is None or anchor2_ring_neighbor is None:
            return None

        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None

        for ridx in sorted(ring_indices, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        anchor_idx1_new = _adjust(anchor_idx1, ring_indices)
        anchor_idx2_new = _adjust(anchor_idx2, ring_indices)

        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol2 = Chem.RWMol(combined)
        offset = rwmol.GetMol().GetNumAtoms()

        frag_attach1 = None
        frag_attach2 = None
        for atom in frag.GetAtoms():
            if atom.GetSymbol() == '*':
                map_num = atom.GetAtomMapNum()
                if map_num == 1:
                    frag_attach1 = atom.GetIdx() + offset
                elif map_num == 2:
                    frag_attach2 = atom.GetIdx() + offset

        if frag_attach1 is None or frag_attach2 is None:
            return None

        dummy1 = rwmol2.GetAtomWithIdx(frag_attach1)
        dummy2 = rwmol2.GetAtomWithIdx(frag_attach2)
        real_neighbor1 = dummy1.GetNeighbors()[0].GetIdx()
        real_neighbor2 = dummy2.GetNeighbors()[0].GetIdx()

        rwmol2.AddBond(anchor_idx1_new, real_neighbor1, Chem.BondType.SINGLE)
        rwmol2.AddBond(anchor_idx2_new, real_neighbor2, Chem.BondType.SINGLE)
        rwmol2.RemoveAtom(max(frag_attach1, frag_attach2))
        rwmol2.RemoveAtom(min(frag_attach1, frag_attach2))

        rwmol = rwmol2
    else:
        return None

    try:
        new_mol = rwmol.GetMol()
        Chem.SanitizeMol(new_mol)
    except Exception:
        return None

    new_smiles = Chem.MolToSmiles(new_mol)

    check_mol = Chem.MolFromSmiles(new_smiles)
    is_valid = check_mol is not None
    if is_valid:
        for atom in check_mol.GetAtoms():
            if (atom.GetNoImplicit() and atom.GetFormalCharge() == 0
                    and atom.GetSymbol() in ('C', 'N', 'O')
                    and atom.GetTotalNumHs() == 0 and atom.GetDegree() < 4):
                is_valid = False
                break

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate["name"],
        "rationale": candidate["rationale"],
        "is_valid": is_valid,
    }

Overwriting src/tools/atom_editor.py


In [13]:
import importlib

import src.tools.replacement_library
import src.tools.atom_editor
import src.tools.molecule_editor

importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)

from src.tools.replacement_library import get_replacement_candidates
from src.tools.atom_editor import apply_atom_edit_from_rule
from src.tools.molecule_editor import propose_fix, find_core_and_target

test_thiol1 = "CC(C)OC(=S)[S-]"
print("thiol_1 fix_test:", propose_fix(test_thiol1, "thiol_1", candidate_idx=0))

print("\n=== 회귀 테스트 ===")
print(propose_fix("CCCCCCC(C)(C)S", "thiol_2", candidate_idx=0))
print(propose_fix("O=C(O)CCl", "alkyl_halide", candidate_idx=0))
print(propose_fix("CC(C)N1C(=O)N(c2ccccc2)CSC1=NC(C)(C)C", "imine_1_general", candidate_idx=0))
print(propose_fix("Cc1cc(NS(=O)(=O)c2ccc(N)cc2)nc(C)n1", "aniline_ring_bcp", candidate_idx=0))

thiol_1 fix_test: {'new_smiles': 'CC(C)OC(N)=O', 'candidate_used': 'carbamate (O,N replacing S,S)', 'rationale': '디티오카바메이트(R-O-C(=S)-S-)를 카바메이트(R-O-C(=O)-N)로 전환. 두 황 원자를 각각 산소·질소로 교체하여 금속 킬레이팅 능력과 효소 억제 활성(디티오카바메이트류 특유의 살충제성 독성 기전)을 제거함 (검증 필요)', 'is_valid': True}

=== 회귀 테스트 ===
{'new_smiles': 'CCCCCCC(C)(C)O', 'candidate_used': 'hydroxyl (alcohol)', 'rationale': '티올의 금속 킬레이팅 및 산화(이황화물/술펜산 형성) 반응성을 제거하면서, 극성·수소결합 특성을 유사하게 유지함', 'is_valid': True}
{'new_smiles': 'O=C(O)CO', 'candidate_used': 'hydroxyl (alcohol)', 'rationale': '이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지', 'is_valid': True}
{'new_smiles': 'CC(C)N1C(=O)N(c2ccccc2)CSC1NC(C)(C)C', 'candidate_used': 'amine (reduced)', 'rationale': '일반 이민(C=N-R)을 환원하여 가수분해 시 반응성 카르보닐로 되돌아갈 수 있는 대사 불안정 경로를 제거함. 옥심 특유의 메커니즘보다는 근거가 다소 약하며, 하위 구조별 개별 검증 필요', 'is_valid': True}
{'new_smiles': 'Cc1cc(NS(=O)(=O)C23CC(N)(C2)C3)nc(C)n1', 'candidate_used': 'BCP (bicyclo[1.1.1]pentane)', 'rationale': 'para-이치환 아닐린의 방향족 벤젠 고리를 포화 bicyclic 탄소골격(BCP)으로 교체함. 방향족성 제거로 

In [14]:
!git add src/tools/replacement_library.py src/tools/atom_editor.py
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   src/tools/atom_editor.py
	modified:   src/tools/replacement_library.py



In [15]:
!git commit -m "Add replace_multi edit type (multiple atoms edited simultaneously with different elements/charges) and thiol_1 rule: converts dithiocarbamate (C(=S)[S-]) to carbamate (C(=O)N) in one atomic edit, removing metal-chelating pesticide-like toxicity mechanism. Library now 14 rules, 6 edit types available."
!git push origin main

[main 5d88009] Add replace_multi edit type (multiple atoms edited simultaneously with different elements/charges) and thiol_1 rule: converts dithiocarbamate (C(=S)[S-]) to carbamate (C(=O)N) in one atomic edit, removing metal-chelating pesticide-like toxicity mechanism. Library now 14 rules, 6 edit types available.
 2 files changed, 28 insertions(+), 10 deletions(-)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 1.40 KiB | 1.40 MiB/s, done.
Total 6 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/Dec32th/laidd-2026.git
   7e8b9b5..5d88009  main -> main


In [16]:
all_rules_now = list(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'].keys())
print("현재 라이브러리:", all_rules_now, f"({len(all_rules_now)}개)\n")

# 각 규칙이 실제로 발동하는 예시 분자를 valid set에서 하나씩 찾아서 실전 테스트
test_examples = {}
for s in data['smiles_valid']:
    problems = detect_toxicophores(s)
    for p in problems:
        if p['rule_name'] in all_rules_now and p['rule_name'] not in test_examples:
            test_examples[p['rule_name']] = s
    if len(test_examples) == len(all_rules_now):
        break

print(f"예시 확보된 규칙: {len(test_examples)}/{len(all_rules_now)}개\n")

print("=== 규칙별 실전 테스트 ===")
for rule, smi in test_examples.items():
    info = get_replacement_candidates(rule)
    all_candidates_ok = True
    for idx in range(len(info['candidates'])):
        result = propose_fix(smi, rule, candidate_idx=idx)
        ok = result is not None and result.get('is_valid', False)
        if not ok:
            all_candidates_ok = False
        status = "✅" if ok else "❌"
        print(f"  {rule} [candidate {idx}]: {status}")
    if not all_candidates_ok:
        print(f"    -> 문제 분자: {smi}")

missing_rules = set(all_rules_now) - set(test_examples.keys())
if missing_rules:
    print(f"\n⚠️ valid set에서 예시를 못 찾은 규칙: {missing_rules}")

현재 라이브러리: ['nitro_group', 'aldehyde', 'Michael_acceptor_1', 'acid_halide', 'alkyl_halide', 'aniline', 'Sulfonic_acid_2', 'imine_1_oxime', 'imine_1_general', 'catechol', 'Thiocarbonyl_group', 'aniline_ring_bcp', 'thiol_2', 'thiol_1'] (14개)

예시 확보된 규칙: 13/14개

=== 규칙별 실전 테스트 ===
  aniline [candidate 0]: ✅
  aniline [candidate 1]: ✅
  nitro_group [candidate 0]: ✅
  nitro_group [candidate 1]: ✅
  nitro_group [candidate 2]: ✅
  imine_1_general [candidate 0]: ✅
  aldehyde [candidate 0]: ✅
  aldehyde [candidate 1]: ✅
  alkyl_halide [candidate 0]: ✅
  alkyl_halide [candidate 1]: ✅
  catechol [candidate 0]: ✅
  Michael_acceptor_1 [candidate 0]: ✅
  Thiocarbonyl_group [candidate 0]: ✅
  acid_halide [candidate 0]: ✅
  acid_halide [candidate 1]: ✅
  Sulfonic_acid_2 [candidate 0]: ✅
  Sulfonic_acid_2 [candidate 1]: ✅
  imine_1_oxime [candidate 0]: ✅
  thiol_2 [candidate 0]: ✅
  thiol_2 [candidate 1]: ✅
  thiol_1 [candidate 0]: ✅

⚠️ valid set에서 예시를 못 찾은 규칙: {'aniline_ring_bcp'}


In [17]:
result_check = detect_toxicophores("Cc1cc(NS(=O)(=O)c2ccc(N)cc2)nc(C)n1")
print([p['rule_name'] for p in result_check])

['aniline']


In [18]:
%%writefile src/tools/toxicophore_detector.py
from rdkit import Chem
from rdkit.Chem import FilterCatalog

def _build_catalog():
    params = FilterCatalog.FilterCatalogParams()
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
    return FilterCatalog.FilterCatalog(params)

_catalog = _build_catalog()
_oxime_pattern = Chem.MolFromSmarts("C=N[OX2H1]")
_para_aniline_pattern = Chem.MolFromSmarts("[NH2]c1ccc([#6,#7,#8,#16])cc1")


def _refine_imine1(mol, atom_indices):
    """imine_1은 옥심(C=N-OH)과 일반 이민(C=N-R)을 모두 포함하는 넓은 카테고리이므로,
    실제 매치된 부분이 옥심 패턴을 포함하는지 확인해 이름을 세분화한다."""
    if mol.HasSubstructMatch(_oxime_pattern):
        matches = mol.GetSubstructMatches(_oxime_pattern)
        for match in matches:
            if set(match) & set(atom_indices):
                return "imine_1_oxime"
    return "imine_1_general"


def detect_toxicophores(smiles: str) -> list[dict]:
    """
    분자의 SMILES를 받아, FilterCatalog(PAINS+BRENK)에 매치되는
    문제 구조(toxicophore)들을 찾아서 규칙 이름과 해당 원자 인덱스를 반환.
    imine_1은 옥심/일반이민 하위형으로 세분화하여 반환한다.
    추가로, para-치환 아닐린(문헌 기반 커스텀 규칙, FilterCatalog 항목 아님)을
    별도로 탐지하여 aniline과 함께 반환한다.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []

    results = []
    for entry in _catalog.GetMatches(mol):
        for fm in entry.GetFilterMatches(mol):
            atom_indices = sorted(set(mol_idx for _, mol_idx in fm.atomPairs))
            rule_name = entry.GetDescription()

            if rule_name == "imine_1":
                rule_name = _refine_imine1(mol, atom_indices)

            results.append({
                "rule_name": rule_name,
                "atom_indices": atom_indices,
            })

    # 커스텀 규칙: para-치환 아닐린 (FilterCatalog에 없는, 문헌 기반 자체 추가 규칙)
    if mol.HasSubstructMatch(_para_aniline_pattern):
        matches = mol.GetSubstructMatches(_para_aniline_pattern)
        for match in matches:
            atom_indices = sorted(set(match))
            results.append({
                "rule_name": "aniline_ring_bcp",
                "atom_indices": atom_indices,
            })

    return results

Overwriting src/tools/toxicophore_detector.py


In [19]:
importlib.reload(src.tools.toxicophore_detector)
from src.tools.toxicophore_detector import detect_toxicophores

result_check2 = detect_toxicophores("Cc1cc(NS(=O)(=O)c2ccc(N)cc2)nc(C)n1")
print([p['rule_name'] for p in result_check2])

['aniline', 'aniline_ring_bcp']


In [20]:
# aniline을 먼저 고친 경우, 그 다음에도 aniline_ring_bcp가 안 나오는지 확인
fixed_aniline_first = propose_fix("Cc1cc(NS(=O)(=O)c2ccc(N)cc2)nc(C)n1", "aniline", candidate_idx=0)
print("aniline 먼저 고침:", fixed_aniline_first['new_smiles'])
print("그 다음 문제들:", [p['rule_name'] for p in detect_toxicophores(fixed_aniline_first['new_smiles'])])

# aniline_ring_bcp를 먼저 고친 경우
fixed_bcp_first = propose_fix("Cc1cc(NS(=O)(=O)c2ccc(N)cc2)nc(C)n1", "aniline_ring_bcp", candidate_idx=0)
print("\nBCP 먼저 고침:", fixed_bcp_first['new_smiles'])
print("그 다음 문제들:", [p['rule_name'] for p in detect_toxicophores(fixed_bcp_first['new_smiles'])])

aniline 먼저 고침: Cc1cc(NS(=O)(=O)c2ccc(C(N)=O)cc2)nc(C)n1
그 다음 문제들: []

BCP 먼저 고침: Cc1cc(NS(=O)(=O)C23CC(N)(C2)C3)nc(C)n1
그 다음 문제들: []


In [21]:
!git add src/tools/toxicophore_detector.py
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   src/tools/toxicophore_detector.py



In [22]:
!git commit -m "Fix critical bug: aniline_ring_bcp was never detected by detect_toxicophores (only defined in replacement_library, no detection logic added) - it would have been a dead rule like phenol/amide/thiourea earlier. Added custom detection for para-substituted aniline. Verified aniline and aniline_ring_bcp are naturally mutually exclusive: fixing either one removes the shared pattern, preventing over-modification of the same amine site."
!git push origin main

[main 5745d44] Fix critical bug: aniline_ring_bcp was never detected by detect_toxicophores (only defined in replacement_library, no detection logic added) - it would have been a dead rule like phenol/amide/thiourea earlier. Added custom detection for para-substituted aniline. Verified aniline and aniline_ring_bcp are naturally mutually exclusive: fixing either one removes the shared pattern, preventing over-modification of the same amine site.
 1 file changed, 14 insertions(+)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 992 bytes | 992.00 KiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   5d88009..5745d44  main -> main


In [23]:
all_rules_final = list(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'].keys())
test_examples_final = {}
for s in data['smiles_valid']:
    problems = detect_toxicophores(s)
    for p in problems:
        if p['rule_name'] in all_rules_final and p['rule_name'] not in test_examples_final:
            test_examples_final[p['rule_name']] = s
    if len(test_examples_final) == len(all_rules_final):
        break

print(f"예시 확보: {len(test_examples_final)}/{len(all_rules_final)}개")
missing = set(all_rules_final) - set(test_examples_final.keys())
if missing:
    print("여전히 못 찾은 규칙:", missing)
else:
    print("14개 규칙 전부 detect_toxicophores로 정상 탐지됨 ✅")

예시 확보: 14/14개
14개 규칙 전부 detect_toxicophores로 정상 탐지됨 ✅


In [5]:
from google.colab import userdata

!pip install openai -q
from openai import OpenAI
dashscope_key = userdata.get('DASHSCOPE_API_KEY')
client_qwen = OpenAI(api_key=dashscope_key, base_url="https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1")
print("Qwen 클라이언트 준비 완료")

Qwen 클라이언트 준비 완료


In [6]:
!pwd
!ls

/content/laidd-2026
data  docs  models  notebooks  outputs	README.md  references  src


In [7]:
from src.tools.molecule_editor import iterative_fix_loop

test_multi_llm = "Cc1cc(NS(=O)(=O)c2ccc(N)cc2)nc(C)n1"
result_llm_test = iterative_fix_loop(
    test_multi_llm, max_iterations=5,
    llm_client=client_qwen, llm_model="qwen3.8-max-preview", llm_client_type="openai_compatible"
)
print("상태:", result_llm_test['status'])
for h in result_llm_test['history']:
    print(h)

상태: success
{'step': 0, 'smiles': 'Cc1cc(NS(=O)(=O)c2ccc(N)cc2)nc(C)n1', 'problems': [{'rule_name': 'aniline', 'atom_indices': [8, 9, 10, 11, 12, 13, 14]}, {'rule_name': 'aniline_ring_bcp', 'atom_indices': [5, 8, 9, 10, 11, 12, 13, 14]}]}
{'step': 1, 'smiles': 'Cc1cc(NS(=O)(=O)c2ccc(C(N)=O)cc2)nc(C)n1', 'fixed_rule': 'aniline', 'problem_reason': '아닐린기는 대사적 N-산화 및 반응성 중간체 형성의 주요 원인이라 먼저 차단하는 것이 독성 완화에 더 직접적이고 화학적으로 타당합니다.', 'candidate_used': 'acetamide (acylated amine)', 'candidate_reason': '아세트아마이드 아실화는 1차 방향족 아민의 N-hydroxylation 경로를 직접 차단하면서도 설폰아미드 주변 극성 및 결합 특성을 비교적 유지할 수 있기 때문입니다.', 'problems': []}


In [9]:
from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use
para_aniline_examples = []
for s in data['smiles_valid']:
    problems = detect_toxicophores(s)
    if any(p['rule_name'] == 'aniline_ring_bcp' for p in problems):
        para_aniline_examples.append(s)
    if len(para_aniline_examples) >= 8:
        break

print(f"테스트할 분자: {len(para_aniline_examples)}개\n")

choice_counts = {"aniline": 0, "aniline_ring_bcp": 0}
for smi in para_aniline_examples:
    problems = detect_toxicophores(smi)
    decision = ask_llm_which_problem_to_fix(client_qwen, "qwen3.8-max-preview", smi, problems, client_type="openai_compatible")
    if decision:
        chosen = decision['rule_name']
        if chosen in choice_counts:
            choice_counts[chosen] += 1
        print(f"{smi[:40]}: {chosen} - {decision.get('reason', '')[:80]}")

print("\n선택 분포:", choice_counts)

테스트할 분자: 8개

Cc1cc(NS(=O)(=O)c2ccc(N)cc2)nc(C)n1: aniline - 아닐린 아민기는 대사 활성화의 직접적 원인이며, 이를 먼저 치환하면 더 큰 aniline_ring_bcp 경고도 함께 제거할 수 있기 때문입니다
Nc1ccc(/N=N\c2ccccc2)c(N)c1: aniline_ring_bcp - 아조기에 파라로 연결된 아닐린 구조는 대사적 N-하이드록실화 및 니트레늄 이온 형성을 통해 돌연변이원성·발암성 위험을 직접 주도하므로, 일반적인
Cc1cc(C)c(N)c(Cl)c1: aniline - 1차 방향족 아민은 대사 활성화로 유전독성 위험을 일으키는 핵심 관능기이므로, 파생된 링 패턴보다 먼저 아민 부분을 수정하는 것이 화학적으로 더
Nc1cc(Cl)c(NC2=NCCN2)c(Cl)c1: aniline - 1차 방향족 아민인 aniline을 먼저 변형하면 근본적인 방향족 아민 독성 경고가 제거되고, 이보다 더 특이적인 aniline_ring_bcp
Cc1cc(-c2cc(C)c(N)c(C)c2)cc(C)c1N: aniline - 일차 방향족 아민 독성포어를 먼저 변형하면 대사적 활성화 위험을 직접 줄이면서 중복된 aniline_ring_bcp 경보도 완화할 수 있기 때문
Nc1ccc(NC(=O)c2ccc(N)cc2)cc1: aniline - 1차 방향족 아민은 N-하이드록실화를 통한 반응성 대사체 형성 위험이 가장 직접적이므로, 고리 수준의 BCP 경고보다 먼저 치환·차단하는 것이 
CC(C)CN(C[C@@H](OP(=O)([O-])[O-])[C@H](C: aniline - 아닐린 아민기는 N-치환 또는 전자끌개 치환으로 직접 차단하기 쉽고, sulfonamide 황까지 포함하는 aniline_ring_bcp보다 먼
COc1cc(N)c(Cl)cc1C(=O)NC1CCN(Cc2ccccc2)C: aniline - 일차 방향족 아민인 아닐린이 대사 활성화로 유전독성 위험을 직접 높이므로, 이 관능기를 먼저 제거하거나 대체하는 것이

In [10]:
%%writefile src/tools/atom_editor.py
from rdkit import Chem


def apply_atom_edit_from_rule(smiles: str, rule_name: str, candidate_idx: int = 0):
    """replacement_library의 atom_edit 규칙을 이용해 원자/결합/고리 직접 편집을 수행.
    candidate마다 다른 edit_type을 가질 수 있음 (예: 같은 문제에 대해
    작은 변화(치환기 하나 추가)와 큰 변화(고리 전체 교체)를 후보로 병렬 제시)."""
    from src.tools.replacement_library import get_replacement_candidates
    info = get_replacement_candidates(rule_name)
    if info is None or info.get("edit_method") != "atom_edit":
        return None
    if candidate_idx >= len(info["candidates"]):
        return None

    candidate = info["candidates"][candidate_idx]
    smarts = info["problem_smarts"]

    mol = Chem.MolFromSmiles(smiles)
    pattern = Chem.MolFromSmarts(smarts)
    if mol is None or pattern is None:
        return None

    matches = mol.GetSubstructMatches(pattern)
    if not matches:
        return None
    match = matches[0]

    rwmol = Chem.RWMol(mol)
    edit_type = candidate["edit_type"]

    if edit_type == "replace_element":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        atom = rwmol.GetAtomWithIdx(target_idx)
        atom.SetAtomicNum(candidate["param"])

    elif edit_type == "add_substituent":
        target_idx = match[candidate.get("target_idx_in_pattern", info.get("target_idx_in_pattern"))]
        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None
        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol = Chem.RWMol(combined)
        offset = mol.GetNumAtoms()
        rwmol.AddBond(target_idx, offset, Chem.BondType.SINGLE)
        atom = rwmol.GetAtomWithIdx(target_idx)
        if atom.GetNumExplicitHs() > 0:
            atom.SetNumExplicitHs(atom.GetNumExplicitHs() - 1)
        else:
            atom.SetNoImplicit(False)

    elif edit_type == "reduce_bond":
        pair = candidate.get("target_idx_pair_in_pattern", info.get("target_idx_pair_in_pattern"))
        idx1 = match[pair[0]]
        idx2 = match[pair[1]]
        bond = rwmol.GetBondBetweenAtoms(idx1, idx2)
        if bond is None:
            return None
        bond.SetBondType(Chem.BondType.SINGLE)
        for idx in (idx1, idx2):
            atom = rwmol.GetAtomWithIdx(idx)
            atom.SetNoImplicit(False)

    elif edit_type == "replace_multi":
        for sub in candidate["param"]:
            target_idx = match[sub["idx_in_pattern"]]
            atom = rwmol.GetAtomWithIdx(target_idx)
            atom.SetAtomicNum(sub["new_element"])
            atom.SetFormalCharge(sub.get("new_charge", 0))
            atom.SetNoImplicit(False)
            atom.SetNumExplicitHs(0)

    elif edit_type == "replace_ring":
        ring_key = candidate.get("ring_atom_indices_in_pattern", info.get("ring_atom_indices_in_pattern"))
        anchor_key = candidate.get("anchor_indices_in_pattern", info.get("anchor_indices_in_pattern"))
        ring_indices = [match[i] for i in ring_key]
        anchor_idx1 = match[anchor_key[0]]
        anchor_idx2 = match[anchor_key[1]]

        anchor1_ring_neighbor = None
        anchor2_ring_neighbor = None
        for ridx in ring_indices:
            ratom = mol.GetAtomWithIdx(ridx)
            neighbor_idxs = [n.GetIdx() for n in ratom.GetNeighbors()]
            if anchor_idx1 in neighbor_idxs:
                anchor1_ring_neighbor = ridx
            if anchor_idx2 in neighbor_idxs:
                anchor2_ring_neighbor = ridx

        if anchor1_ring_neighbor is None or anchor2_ring_neighbor is None:
            return None

        frag = Chem.MolFromSmiles(candidate["param"])
        if frag is None:
            return None

        for ridx in sorted(ring_indices, reverse=True):
            rwmol.RemoveAtom(ridx)

        def _adjust(idx, removed):
            shift = sum(1 for r in removed if r < idx)
            return idx - shift

        anchor_idx1_new = _adjust(anchor_idx1, ring_indices)
        anchor_idx2_new = _adjust(anchor_idx2, ring_indices)

        combined = Chem.CombineMols(rwmol.GetMol(), frag)
        rwmol2 = Chem.RWMol(combined)
        offset = rwmol.GetMol().GetNumAtoms()

        frag_attach1 = None
        frag_attach2 = None
        for atom in frag.GetAtoms():
            if atom.GetSymbol() == '*':
                map_num = atom.GetAtomMapNum()
                if map_num == 1:
                    frag_attach1 = atom.GetIdx() + offset
                elif map_num == 2:
                    frag_attach2 = atom.GetIdx() + offset

        if frag_attach1 is None or frag_attach2 is None:
            return None

        dummy1 = rwmol2.GetAtomWithIdx(frag_attach1)
        dummy2 = rwmol2.GetAtomWithIdx(frag_attach2)
        real_neighbor1 = dummy1.GetNeighbors()[0].GetIdx()
        real_neighbor2 = dummy2.GetNeighbors()[0].GetIdx()

        rwmol2.AddBond(anchor_idx1_new, real_neighbor1, Chem.BondType.SINGLE)
        rwmol2.AddBond(anchor_idx2_new, real_neighbor2, Chem.BondType.SINGLE)
        rwmol2.RemoveAtom(max(frag_attach1, frag_attach2))
        rwmol2.RemoveAtom(min(frag_attach1, frag_attach2))

        rwmol = rwmol2
    else:
        return None

    try:
        new_mol = rwmol.GetMol()
        Chem.SanitizeMol(new_mol)
    except Exception:
        return None

    new_smiles = Chem.MolToSmiles(new_mol)

    check_mol = Chem.MolFromSmiles(new_smiles)
    is_valid = check_mol is not None
    if is_valid:
        for atom in check_mol.GetAtoms():
            if (atom.GetNoImplicit() and atom.GetFormalCharge() == 0
                    and atom.GetSymbol() in ('C', 'N', 'O')
                    and atom.GetTotalNumHs() == 0 and atom.GetDegree() < 4):
                is_valid = False
                break

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate["name"],
        "rationale": candidate["rationale"],
        "is_valid": is_valid,
    }

Overwriting src/tools/atom_editor.py


In [11]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "Michael_acceptor_1": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=CC(=O)",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "saturated (C-C single bond)",
             "rationale": "알파,베타-불포화 카르보닐의 C=C 이중결합을 환원하여 "
                          "단백질 친전자성 부가반응(Michael addition, covalent "
                          "binding) 위험을 제거함"},
        ],
    },
    "acid_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
            {"smiles": "C(=O)O", "name": "ester",
             "rationale": "아마이드보다 극성이 낮고 유연한 대체 옵션, 가수분해 속도 조절 가능 (검증 필요)"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "이탈기를 제거해 알킬화 반응성을 없앰, 극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NH2]c1ccc([#6,#7,#8,#16])cc1",
        "target_idx_in_pattern": 0,
        "ring_atom_indices_in_pattern": [1, 2, 3, 4, 6, 7],
        "anchor_indices_in_pattern": (0, 5),
        "candidates": [
            {"edit_type": "add_substituent", "param": "C(=O)C",
             "target_idx_in_pattern": 0,
             "name": "acetamide (acylated amine)",
             "rationale": "1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"edit_type": "replace_ring", "param": "[*:1]C12CC(C1)(C2)[*:2]",
             "ring_atom_indices_in_pattern": [1, 2, 3, 4, 6, 7],
             "anchor_indices_in_pattern": (0, 5),
             "name": "BCP (bicyclo[1.1.1]pentane)",
             "rationale": "para-이치환 아닐린의 방향족 벤젠 고리를 포화 bicyclic "
                          "탄소골격(BCP)으로 교체함. 방향족성 제거로 aniline reactive "
                          "metabolite(RM) 형성 및 CYP-inhibition을 감소시켜, 퀴논이민 "
                          "생성 경로를 차단하고 특이체질 약물 부작용(IADR) 위험을 낮춤 "
                          "(문헌 근거, 학생 제공). 벤젠과의 공간적 유사성, Fsp3 증가, "
                          "실제 성공 사례가 많아 채택. 아마이드화(단순 아민 치환)보다 "
                          "변화 폭이 크지만, 물성 개선 효과도 더 큼"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "S(=O)(=O)[OX2H1,OX1-]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "생리적 pH에서 이온화 정도(전하)를 크게 낮춰 세포막 투과성을 "
                          "개선함. 설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 "
                          "저해되는 경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
            {"smiles": "C(=O)O", "name": "carboxylic acid",
             "rationale": "설폰산보다 산성도가 약하고 부피가 작은 산성 bioisostere "
                          "(검증 필요)"},
        ],
    },
    "imine_1_oxime": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=N[OX2H1]",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
        ],
    },
    "imine_1_general": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=N",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "일반 이민(C=N-R)을 환원하여 가수분해 시 반응성 카르보닐로 "
                          "되돌아갈 수 있는 대사 불안정 경로를 제거함. 옥심 특유의 "
                          "메커니즘보다는 근거가 다소 약하며, 하위 구조별 개별 검증 필요"},
        ],
    },
    "catechol": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2H;$(Oc1ccccc1O)]",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "add_substituent", "param": "C", "name": "methoxy",
             "rationale": "인체의 COMT(catechol-O-methyltransferase) 효소가 카테콜을 "
                          "메톡시페놀로 메틸화하여 해독하는 생리적 경로와 동일한 원리. "
                          "오르토-퀴논으로의 산화 경로를 차단하여 세포독성/유전독성 우려를 "
                          "낮춤 (학생 확인 예정: ScienceDirect catechol overview, "
                          "PMC6643002 등 참고)"},
        ],
    },
    "Thiocarbonyl_group": {
        "edit_method": "atom_edit",
        "problem_smarts": "[#6]=[#16]",
        "target_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "carbonyl (O replacing S)",
             "rationale": "황을 산소로 대체(티오카르보닐->카르보닐)하는 것은 흔한 "
                          "bioisostere 전략으로, 갑상선 기능 저해 등 황 함유 작용기 "
                          "특유의 대사/독성 우려를 낮춤 (검증 필요, thiourea->urea "
                          "치환 논리와 동일 계열)"},
        ],
    },
    "thiol_2": {
        "problem_smarts": "[SX2H1]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "티올의 금속 킬레이팅 및 산화(이황화물/술펜산 형성) 반응성을 "
                          "제거하면서, 극성·수소결합 특성을 유사하게 유지함"},
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "티올을 아마이드로 대체하여 반응성을 낮추면서 약물유사 골격에서 "
                          "흔히 쓰이는 안정적 작용기로 전환 (검증 필요)"},
        ],
    },
    "thiol_1": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=S)[SX1-]",
        "candidates": [
            {"edit_type": "replace_multi",
             "param": [
                 {"idx_in_pattern": 1, "new_element": 8, "new_charge": 0},
                 {"idx_in_pattern": 2, "new_element": 7, "new_charge": 0},
             ],
             "name": "carbamate (O,N replacing S,S)",
             "rationale": "디티오카바메이트(R-O-C(=S)-S-)를 카바메이트(R-O-C(=O)-N)로 "
                          "전환. 두 황 원자를 각각 산소·질소로 교체하여 금속 킬레이팅 "
                          "능력과 효소 억제 활성(디티오카바메이트류 특유의 살충제성 "
                          "독성 기전)을 제거함 (검증 필요)"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)

Overwriting src/tools/replacement_library.py


In [12]:
%%writefile src/tools/toxicophore_detector.py
from rdkit import Chem
from rdkit.Chem import FilterCatalog

def _build_catalog():
    params = FilterCatalog.FilterCatalogParams()
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
    return FilterCatalog.FilterCatalog(params)

_catalog = _build_catalog()
_oxime_pattern = Chem.MolFromSmarts("C=N[OX2H1]")


def _refine_imine1(mol, atom_indices):
    """imine_1은 옥심(C=N-OH)과 일반 이민(C=N-R)을 모두 포함하는 넓은 카테고리이므로,
    실제 매치된 부분이 옥심 패턴을 포함하는지 확인해 이름을 세분화한다."""
    if mol.HasSubstructMatch(_oxime_pattern):
        matches = mol.GetSubstructMatches(_oxime_pattern)
        for match in matches:
            if set(match) & set(atom_indices):
                return "imine_1_oxime"
    return "imine_1_general"


def detect_toxicophores(smiles: str) -> list[dict]:
    """
    분자의 SMILES를 받아, FilterCatalog(PAINS+BRENK)에 매치되는
    문제 구조(toxicophore)들을 찾아서 규칙 이름과 해당 원자 인덱스를 반환.
    imine_1은 옥심/일반이민 하위형으로 세분화하여 반환한다.
    aniline은 FilterCatalog의 단순 [NH2] 탐지 대신, replacement_library의
    확장된 패턴(para-치환 벤젠 포함)을 그대로 사용해 재정의한다.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []

    results = []
    for entry in _catalog.GetMatches(mol):
        for fm in entry.GetFilterMatches(mol):
            atom_indices = sorted(set(mol_idx for _, mol_idx in fm.atomPairs))
            rule_name = entry.GetDescription()

            if rule_name == "imine_1":
                rule_name = _refine_imine1(mol, atom_indices)
            elif rule_name == "aniline":
                continue  # 아래에서 확장된 패턴으로 다시 탐지하므로 원본은 건너뜀

            results.append({
                "rule_name": rule_name,
                "atom_indices": atom_indices,
            })

    # aniline: 원래 FilterCatalog의 단순 [NH2] 대신, replacement_library와
    # 정확히 일치하는 확장된 SMARTS로 재탐지 (아민만 vs 고리 전체, 두 candidate가
    # 모두 이 하나의 매치 위에서 작동하도록 통일)
    from src.tools.replacement_library import get_replacement_candidates
    aniline_info = get_replacement_candidates("aniline")
    if aniline_info:
        aniline_pattern = Chem.MolFromSmarts(aniline_info["problem_smarts"])
        if mol.HasSubstructMatch(aniline_pattern):
            matches = mol.GetSubstructMatches(aniline_pattern)
            for match in matches:
                atom_indices = sorted(set(match))
                results.append({
                    "rule_name": "aniline",
                    "atom_indices": atom_indices,
                })

    return results

Overwriting src/tools/toxicophore_detector.py


In [13]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.toxicophore_detector)
importlib.reload(src.tools.molecule_editor)
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.molecule_editor import propose_fix

test_aniline = "Cc1cc(NS(=O)(=O)c2ccc(N)cc2)nc(C)n1"
print("문제 목록:", [p['rule_name'] for p in detect_toxicophores(test_aniline)])
print("\ncandidate 0 (아세트아미드):", propose_fix(test_aniline, "aniline", candidate_idx=0))
print("\ncandidate 1 (BCP):", propose_fix(test_aniline, "aniline", candidate_idx=1))

문제 목록: ['aniline']

candidate 0 (아세트아미드): {'new_smiles': 'CC(=O)Nc1ccc(S(=O)(=O)Nc2cc(C)nc(C)n2)cc1', 'candidate_used': 'acetamide (acylated amine)', 'rationale': '1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단', 'is_valid': True}

candidate 1 (BCP): {'new_smiles': 'Cc1cc(NS(=O)(=O)C23CC(N)(C2)C3)nc(C)n1', 'candidate_used': 'BCP (bicyclo[1.1.1]pentane)', 'rationale': 'para-이치환 아닐린의 방향족 벤젠 고리를 포화 bicyclic 탄소골격(BCP)으로 교체함. 방향족성 제거로 aniline reactive metabolite(RM) 형성 및 CYP-inhibition을 감소시켜, 퀴논이민 생성 경로를 차단하고 특이체질 약물 부작용(IADR) 위험을 낮춤 (문헌 근거, 학생 제공). 벤젠과의 공간적 유사성, Fsp3 증가, 실제 성공 사례가 많아 채택. 아마이드화(단순 아민 치환)보다 변화 폭이 크지만, 물성 개선 효과도 더 큼', 'is_valid': True}


In [14]:
decision_test = ask_llm_which_candidate_to_use(client_qwen, "qwen3.8-max-preview", test_aniline, "aniline", client_type="openai_compatible")
print(decision_test)

{'candidate_idx': 1, 'reason': 'para-아닐린 벤젠 고리를 BCP로 교체하면 방향족성을 제거해 퀴논이민형 반응성 대사체 생성을 근본적으로 차단하면서도 치환 벡터와 물성을 개선할 수 있기 때문입니다.'}


In [15]:
para_aniline_examples2 = []
for s in data['smiles_valid']:
    problems = detect_toxicophores(s)
    if any(p['rule_name'] == 'aniline' for p in problems):
        info = get_replacement_candidates("aniline")
        mol = Chem.MolFromSmiles(s)
        if mol.HasSubstructMatch(Chem.MolFromSmarts(info['problem_smarts'])):
            para_aniline_examples2.append(s)
    if len(para_aniline_examples2) >= 8:
        break

print(f"테스트할 분자: {len(para_aniline_examples2)}개\n")

choice_counts2 = {0: 0, 1: 0}
for smi in para_aniline_examples2:
    decision = ask_llm_which_candidate_to_use(client_qwen, "qwen3.8-max-preview", smi, "aniline", client_type="openai_compatible")
    if decision:
        idx = decision['candidate_idx']
        choice_counts2[idx] = choice_counts2.get(idx, 0) + 1
        name = "아세트아미드" if idx == 0 else "BCP"
        print(f"{smi[:40]}: {name} - {decision.get('reason', '')[:80]}")

print("\n선택 분포 (0=아세트아미드, 1=BCP):", choice_counts2)

테스트할 분자: 8개

Cc1cc(NS(=O)(=O)c2ccc(N)cc2)nc(C)n1: BCP - para-이치환 아닐린 벤젠 고리를 BCP로 교체하면 아닐린 특유의 N-hydroxylation/퀴논이민 반응성 대사체 경로를 근본적으로 차단하
Nc1ccc(/N=N\c2ccccc2)c(N)c1: BCP - BCP는 아닐린 방향족 고리를 포화 골격으로 대체하여 N-hydroxylation 및 퀴논이민 형성 경로를 근본적으로 차단하고 Fsp3 증가로 
Cc1cc(C)c(N)c(Cl)c1: 아세트아미드 - BCP는 ortho 치환기가 있는 다치환 아닐린에는 기존 벤젠 치환 패턴을 보존하기 어렵지만, 아세트아마이드화는 구조와 결합 모드를 유지하면서 
Nc1cc(Cl)c(NC2=NCCN2)c(Cl)c1: 아세트아미드 - 1차 방향족 아민을 아세타마이드로 아실화하면 아닐린의 N-하이드록실화 및 반응성 대사체 생성을 직접 차단하면서도 기존 분자 골격과 치환 패턴을 
Cc1cc(-c2cc(C)c(N)c(C)c2)cc(C)c1N: BCP - 이 분자는 두 개의 para-이치환 2,6-dimethyl aniline 고리를 가지므로 BCP로 방향족 고리를 대체하면 아닐린 기반 N-산화/
Nc1ccc(NC(=O)c2ccc(N)cc2)cc1: BCP - para-이치환 아닐린 고리를 BCP로 교체하면 방향족 아민 대사활성화와 퀴논이민 RM 형성 위험을 근본적으로 줄이고 Fsp3와 물성도 개선할 
CC(C)CN(C[C@@H](OP(=O)([O-])[O-])[C@H](C: BCP - para-이치환 아닐린의 벤젠 고리를 BCP로 교체하면 방향족 N-하이드록실화 및 퀴논이민 생성 경로를 근본적으로 차단하면서 벡터 유사성과 Fs
COc1cc(N)c(Cl)cc1C(=O)NC1CCN(Cc2ccccc2)C: 아세트아미드 - 1차 방향족 아민을 아세타마이드로 아실화하면 기존 치환 패턴과 분자 골격을 유지하면서 N-hydroxylation 및 퀴논이민 생성 경로를 직접

선택 분포 (0=아세트아미드

In [16]:
!git add src/tools/replacement_library.py src/tools/atom_editor.py src/tools/toxicophore_detector.py
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   src/tools/atom_editor.py
	modified:   src/tools/replacement_library.py
	modified:   src/tools/toxicophore_detector.py



In [17]:
!git commit -m "Merge aniline and aniline_ring_bcp into single 'aniline' rule with two candidates (acetamide vs BCP ring replacement). Fixes severe LLM bias discovered via 8-molecule test: 'problem priority' framing caused 7/8 selection of conservative option regardless of context; 'candidate selection' framing after merge yields balanced 5:3 split with molecule-specific reasoning (e.g. correctly avoiding BCP for ortho-substituted anilines where ring geometry doesn't transfer well). Library now 13 rules (aniline_ring_bcp removed as standalone)."
!git push origin main

[main 0d71890] Merge aniline and aniline_ring_bcp into single 'aniline' rule with two candidates (acetamide vs BCP ring replacement). Fixes severe LLM bias discovered via 8-molecule test: 'problem priority' framing caused 7/8 selection of conservative option regardless of context; 'candidate selection' framing after merge yields balanced 5:3 split with molecule-specific reasoning (e.g. correctly avoiding BCP for ortho-substituted anilines where ring geometry doesn't transfer well). Library now 13 rules (aniline_ring_bcp removed as standalone).
 3 files changed, 52 insertions(+), 43 deletions(-)
Enumerating objects: 13, done.
Counting objects: 100% (13/13), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 1.95 KiB | 1.96 MiB/s, done.
Total 7 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), completed with 5 local objects.
To https://github.com/Dec32th/laidd-2026.git
   6cdc3d6..0d71890  mai